# 스트레스 점수 예측 V18

**V14 + XGBoost/LightGBM Complementary Ensemble**

- 1등 V14 Blend95를 최소 70% 유지
- XGBoost와 LightGBM을 MAE 목적함수로 학습
- XGBoost, LightGBM, 50:50 Boosting 앙상블을 각각 비교
- 서로 다른 두 5-Fold 분할 모두에서 V14를 이긴 조합만 선택
- Test 데이터는 전처리 학습과 비중 선택에 사용하지 않음

In [1]:
from pathlib import Path
import os

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

## 1. 설정

In [2]:
DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project")

TARGET = "stress_score"
ID_COLUMN = "ID"
MODEL_SEEDS = (42, 77, 2026)
CV_SEEDS = (42, 2026)
N_ESTIMATORS = 1200
BASE_QUANTILE = 0.510
POOLED_QUANTILE = 0.505
AGE_QUANTILE = 0.500
BOOST_WEIGHTS = np.round(np.arange(0.00, 0.31, 0.05), 2)
BOOST_MIXES = ("xgb", "lgb", "mean")

WEIGHTED_FEATURES = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio",
    "bone_density",
]
AGE_WEIGHTED_FEATURES = WEIGHTED_FEATURES + [
    "mean_arterial_pressure", "diastolic_blood_pressure",
]

## 2. 데이터와 파생변수

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test 입력 컬럼이 다릅니다.")
if not submission[ID_COLUMN].equals(test[ID_COLUMN]):
    raise ValueError("제출 양식과 Test의 ID 순서가 다릅니다.")


def add_base_features(frame):
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = (
        result["cholesterol"] / (result["glucose"] + 1e-6)
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_weighted_features(frame):
    result = add_base_features(frame)
    for feature in WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
        result[f"{feature}__copy2"] = result[feature]
    return result


def add_age_features(frame):
    result = add_base_features(frame)
    result["age_bmi"] = result["age"] * result["bmi"]
    result["age_glucose"] = result["age"] * result["glucose"]
    result["age_cholesterol"] = result["age"] * result["cholesterol"]
    result["age_bone_density"] = result["age"] * result["bone_density"]
    result["age_mean_arterial_pressure"] = (
        result["age"] * result["mean_arterial_pressure"]
    )
    for feature in AGE_WEIGHTED_FEATURES:
        result[f"{feature}__copy1"] = result[feature]
    return result


y = train[TARGET].reset_index(drop=True)
raw_train = train.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
raw_test = test.drop(columns=[ID_COLUMN]).reset_index(drop=True)
X_weighted = add_weighted_features(raw_train)
X_test_weighted = add_weighted_features(raw_test)
X_age = add_age_features(raw_train)
X_test_age = add_age_features(raw_test)
X_boost = add_base_features(raw_train)
X_test_boost = add_base_features(raw_test)
print("Train:", train.shape, "| Boost features:", X_boost.shape)

Train: (3000, 18) | Boost features: (3000, 22)


## 3. Train 전용 전처리

In [4]:
def make_preprocessor(frame):
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="constant",
                                fill_value="missing",
                            ),
                        ),
                        (
                            "encoder",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False,
                            ),
                        ),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_matrix(model, matrix):
    return np.column_stack(
        [tree.predict(matrix) for tree in model.estimators_]
    )


def q(matrix, quantile):
    return np.clip(
        np.quantile(matrix, quantile, axis=1),
        0.0,
        1.0,
    )

## 4. Repeated 5-Fold OOF 예측

In [5]:
split_oof = {}
xgb_iterations = []
lgb_iterations = []

for cv_seed in CV_SEEDS:
    cv = KFold(n_splits=5, shuffle=True, random_state=cv_seed)
    oof_v14 = np.zeros(len(y))
    oof_xgb = np.zeros(len(y))
    oof_lgb = np.zeros(len(y))

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_weighted), start=1):
        weighted_prep = make_preprocessor(X_weighted.iloc[train_idx])
        wt = weighted_prep.fit_transform(X_weighted.iloc[train_idx])
        wv = weighted_prep.transform(X_weighted.iloc[valid_idx])
        matrices = {}
        for seed in MODEL_SEEDS:
            model = ExtraTreesRegressor(
                n_estimators=N_ESTIMATORS, min_samples_leaf=1,
                max_features=1, random_state=seed, n_jobs=-1,
            )
            model.fit(wt, y.iloc[train_idx])
            matrices[seed] = tree_matrix(model, wv)
        pred_base = q(matrices[42], BASE_QUANTILE)
        pred_pooled = q(
            np.concatenate([matrices[s] for s in MODEL_SEEDS], axis=1),
            POOLED_QUANTILE,
        )

        age_prep = make_preprocessor(X_age.iloc[train_idx])
        at = age_prep.fit_transform(X_age.iloc[train_idx])
        av = age_prep.transform(X_age.iloc[valid_idx])
        age_model = ExtraTreesRegressor(
            n_estimators=N_ESTIMATORS, min_samples_leaf=1,
            max_features=1, random_state=42, n_jobs=-1,
        )
        age_model.fit(at, y.iloc[train_idx])
        pred_age = q(tree_matrix(age_model, av), AGE_QUANTILE)
        pred_aux = np.clip(0.70 * pred_pooled + 0.30 * pred_age, 0, 1)
        oof_v14[valid_idx] = np.clip(
            0.95 * pred_base + 0.05 * pred_aux, 0, 1
        )

        boost_prep = make_preprocessor(X_boost.iloc[train_idx])
        bt = boost_prep.fit_transform(X_boost.iloc[train_idx])
        bv = boost_prep.transform(X_boost.iloc[valid_idx])

        xgb = XGBRegressor(
            objective="reg:absoluteerror", eval_metric="mae",
            n_estimators=3000, learning_rate=0.02, max_depth=4,
            min_child_weight=5, subsample=0.8, colsample_bytree=0.8,
            reg_lambda=5.0, reg_alpha=0.1,
            random_state=42 + fold, n_jobs=-1,
            early_stopping_rounds=150,
        )
        xgb.fit(bt, y.iloc[train_idx], eval_set=[(bv, y.iloc[valid_idx])], verbose=False)
        oof_xgb[valid_idx] = np.clip(xgb.predict(bv), 0, 1)
        xgb_iterations.append(xgb.best_iteration + 1)

        lgb = LGBMRegressor(
            objective="regression_l1", metric="mae",
            n_estimators=3000, learning_rate=0.02, num_leaves=15,
            min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
            reg_lambda=5.0, reg_alpha=0.1,
            random_state=42 + fold, n_jobs=-1, verbosity=-1,
        )
        lgb.fit(
            bt, y.iloc[train_idx], eval_set=[(bv, y.iloc[valid_idx])],
            callbacks=[early_stopping(150, verbose=False), log_evaluation(0)],
        )
        oof_lgb[valid_idx] = np.clip(lgb.predict(bv), 0, 1)
        lgb_iterations.append(lgb.best_iteration_)
        print(f"CV seed {cv_seed} | Fold {fold}: complete")

    split_oof[cv_seed] = {"v14": oof_v14, "xgb": oof_xgb, "lgb": oof_lgb}
    print(
        f"CV seed {cv_seed} | V14={mean_absolute_error(y,oof_v14):.6f} | "
        f"XGB={mean_absolute_error(y,oof_xgb):.6f} | "
        f"LGB={mean_absolute_error(y,oof_lgb):.6f}"
    )

/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


CV seed 42 | Fold 1: complete


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


CV seed 42 | Fold 2: complete


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

CV seed 2026 | Fold 3: complete


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


CV seed 2026 | Fold 4: complete


/opt/anaconda3/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


CV seed 2026 | Fold 5: complete
CV seed 2026 | V14=0.145586 | XGB=0.212586 | LGB=0.211010


## 5. 안정적인 모델·비중 선택

In [6]:
def boost_prediction(split, mix):
    if mix == "xgb": return split["xgb"]
    if mix == "lgb": return split["lgb"]
    return 0.5 * split["xgb"] + 0.5 * split["lgb"]


rows = []
for mix in BOOST_MIXES:
    for boost_weight in BOOST_WEIGHTS:
        scores, improvements = [], []
        for cv_seed in CV_SEEDS:
            split = split_oof[cv_seed]
            boost = boost_prediction(split, mix)
            pred = np.clip(
                (1.0 - boost_weight) * split["v14"] + boost_weight * boost,
                0, 1,
            )
            base_mae = mean_absolute_error(y, split["v14"])
            score = mean_absolute_error(y, pred)
            scores.append(score)
            improvements.append(base_mae - score)
        rows.append({
            "boost_mix": mix, "boost_weight": boost_weight,
            "v14_weight": 1.0 - boost_weight,
            "mean_CV_MAE": np.mean(scores),
            "mean_improvement": np.mean(improvements),
            "worst_improvement": min(improvements),
            "wins": sum(v > 0 for v in improvements),
        })

results = pd.DataFrame(rows).sort_values(
    ["wins", "worst_improvement", "mean_CV_MAE"],
    ascending=[False, False, True],
).reset_index(drop=True)
eligible = results[(results.wins == len(CV_SEEDS)) & (results.worst_improvement > 0)]
selected = eligible.iloc[0] if len(eligible) else results[results.boost_weight == 0].iloc[0]
SELECTED_MIX = selected["boost_mix"]
SELECTED_BOOST_WEIGHT = float(selected["boost_weight"])
SELECTED_V14_WEIGHT = 1.0 - SELECTED_BOOST_WEIGHT
FINAL_XGB_ITERATIONS = int(np.median(xgb_iterations))
FINAL_LGB_ITERATIONS = int(np.median(lgb_iterations))

print("Selected boost mix:", SELECTED_MIX)
print("Selected V14 weight:", SELECTED_V14_WEIGHT)
print("Selected boosting weight:", SELECTED_BOOST_WEIGHT)
print("Repeated-CV mean MAE:", f"{selected['mean_CV_MAE']:.6f}")
print("Worst split improvement:", f"{selected['worst_improvement']:+.6f}")
print("Wins across CV splits:", int(selected["wins"]), "/", len(CV_SEEDS))
results.head(15)

Selected boost mix: xgb
Selected V14 weight: 1.0
Selected boosting weight: 0.0
Repeated-CV mean MAE: 0.147333
Worst split improvement: +0.000000
Wins across CV splits: 0 / 2


,boost_mix,boost_weight,v14_weight,mean_CV_MAE,mean_improvement,worst_improvement,wins
0,xgb,0.00,1.00,0.147333,0.000000,0.000000,0
1,lgb,0.00,1.00,0.147333,0.000000,0.000000,0
2,mean,0.00,1.00,0.147333,0.000000,0.000000,0
3,lgb,0.05,0.95,0.150222,-0.002889,-0.002994,0
4,mean,0.05,0.95,0.150263,-0.002930,-0.003023,0
5,xgb,0.05,0.95,0.150330,-0.002997,-0.003079,0
6,lgb,0.10,0.90,0.153139,-0.005807,-0.006006,0
7,mean,0.10,0.90,0.153215,-0.005882,-0.006057,0
8,xgb,0.10,0.90,0.153352,-0.006019,-0.006171,0
9,lgb,0.15,0.85,0.156091,-0.008758,-0.009037,0


## 6. 전체 Train 학습과 Test 예측

In [7]:
# V14 재현
wp = make_preprocessor(X_weighted)
wt = wp.fit_transform(X_weighted); wtest = wp.transform(X_test_weighted)
matrices = {}
for seed in MODEL_SEEDS:
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1,
        max_features=1, random_state=seed, n_jobs=-1,
    )
    model.fit(wt, y); matrices[seed] = tree_matrix(model, wtest)
pred_base = q(matrices[42], BASE_QUANTILE)
pred_pooled = q(np.concatenate([matrices[s] for s in MODEL_SEEDS], axis=1), POOLED_QUANTILE)
ap = make_preprocessor(X_age)
at = ap.fit_transform(X_age); atest = ap.transform(X_test_age)
am = ExtraTreesRegressor(n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1, random_state=42, n_jobs=-1)
am.fit(at, y); pred_age = q(tree_matrix(am, atest), AGE_QUANTILE)
pred_v14 = np.clip(0.95 * pred_base + 0.05 * (0.70 * pred_pooled + 0.30 * pred_age), 0, 1)

if SELECTED_BOOST_WEIGHT > 0:
    bp = make_preprocessor(X_boost)
    bt = bp.fit_transform(X_boost); btest = bp.transform(X_test_boost)
    if SELECTED_MIX in ("xgb", "mean"):
        xgb = XGBRegressor(
            objective="reg:absoluteerror", n_estimators=FINAL_XGB_ITERATIONS,
            learning_rate=0.02, max_depth=4, min_child_weight=5,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0,
            reg_alpha=0.1, random_state=42, n_jobs=-1,
        )
        xgb.fit(bt, y); pred_xgb = np.clip(xgb.predict(btest), 0, 1)
    if SELECTED_MIX in ("lgb", "mean"):
        lgb = LGBMRegressor(
            objective="regression_l1", n_estimators=FINAL_LGB_ITERATIONS,
            learning_rate=0.02, num_leaves=15, min_child_samples=30,
            subsample=0.8, colsample_bytree=0.8, reg_lambda=5.0,
            reg_alpha=0.1, random_state=42, n_jobs=-1, verbosity=-1,
        )
        lgb.fit(bt, y); pred_lgb = np.clip(lgb.predict(btest), 0, 1)
    if SELECTED_MIX == "xgb": pred_boost = pred_xgb
    elif SELECTED_MIX == "lgb": pred_boost = pred_lgb
    else: pred_boost = 0.5 * pred_xgb + 0.5 * pred_lgb
else:
    pred_boost = pred_v14.copy()

pred_v18 = np.clip(
    SELECTED_V14_WEIGHT * pred_v14 + SELECTED_BOOST_WEIGHT * pred_boost,
    0, 1,
)
print("Prediction range:", pred_v18.min(), pred_v18.max())
print("첫 10개 예측값:", pred_v18[:10])

Prediction range: 0.0 1.0
첫 10개 예측값: [0.56055 0.97    0.19    0.44015 0.53    0.46    0.78    0.4697  0.5505
 0.88   ]


## 7. 제출 파일

In [8]:
submission_v18 = submission.copy()
submission_v18[ID_COLUMN] = test[ID_COLUMN]
submission_v18[TARGET] = pred_v18
output_path = OUTPUT_DIR / "submit_v18_v14_boosting.csv"
submission_v18.to_csv(output_path, index=False, encoding="utf-8")
print("Saved:", output_path)
submission_v18.head(10)

Saved: /Users/bitsaem/Desktop/stress_project/submit_v18_v14_boosting.csv


,ID,stress_score
0,TEST_0000,0.56055
1,TEST_0001,0.97000
2,TEST_0002,0.19000
3,TEST_0003,0.44015
4,TEST_0004,0.53000
5,TEST_0005,0.46000
6,TEST_0006,0.78000
7,TEST_0007,0.46970
8,TEST_0008,0.55050
9,TEST_0009,0.88000


## 제출 기준

- `Wins across CV splits: 2 / 2`
- `Worst split improvement > 0`
- Boosting 비중이 0보다 클 때만 제출 후보
- 조건을 만족하지 못하면 V14 Blend95 유지